# 🎬 SRT Subtitle Translator — Ollama Local
### English → Brazilian Portuguese | 100% offline

**Prerequisites:**
- Ollama installed and running
- Model downloaded: `ollama pull translategemma:12b`
- File `tradutor_legendas.py` in the same folder as this notebook

**How to use:**
1. Make sure Ollama is running (`ollama serve`)
2. Run the cells in order
3. Set the path to the `.srt` file and the style in **Cell 2**
4. Run **Cell 3** to translate
5. Run **Cell 4** *(optional)* for final review + automatic correction

> ♻️ **Automatic checkpoint:** if the process is interrupted, just run Cell 3 again — it resumes from where it left off.

---

### 🔄 Translation modes

| Mode | `USE_GOOGLE_AS_BASE` | Flow |
|------|----------------------|------|
| **Hybrid** *(recommended)* | `True` | Google Translate → review/naturalization by local model |
| **Pure model** | `False` | Translation + review done entirely by local model |

---

| Style | Code | Best for |
|-------|------|----------|
| 🎬 Cinema | `1` | Movies and series in general |
| 💬 Colloquial | `2` | Everyday conversations |
| 🎓 Formal | `3` | Documentaries and educational content |
| 😄 Casual | `4` | Comedies, reality shows and vlogs |
| 🔬 Academic | `5` | Scientific documentaries, technical terms |


In [ ]:
# ✅ Cell 1 — Check and install dependencies
import subprocess, sys, shutil

for package, module in [('ollama', 'ollama'), ('deep-translator', 'deep_translator')]:
    try:
        __import__(module)
        print(f'✅ {package} already installed.')
    except ImportError:
        print(f'📥 Installing {package}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f'✅ {package} installed.')

if shutil.which('ollama'):
    print('✅ Ollama installed on the system.')
else:
    import platform
    system = platform.system()
    print(f'⚠️  Ollama not found. System: {system}')
    if system == 'Windows':
        import urllib.request, os, tempfile
        url = 'https://ollama.com/download/OllamaSetup.exe'
        installer = os.path.join(tempfile.gettempdir(), 'OllamaSetup.exe')
        urllib.request.urlretrieve(url, installer)
        subprocess.Popen([installer])
        print('   ⚠️  After installation, restart the kernel and run this cell again.')
    elif system == 'Linux':
        subprocess.check_call(['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'])
        print('✅ Ollama installed. Run "ollama serve" in a separate terminal.')
    else:
        print('   Download and install manually: https://ollama.com/download')


In [ ]:
# ✅ Cell 2 — Settings
import ollama
from deep_translator import GoogleTranslator
from functions.translation_process import STYLES

# ─────────────────────────────────────────────────────────
# 🤖 MODELS
# translategemma:12b  → great for EN→PT-BR translation
# gemma3:12b          → good general alternative
# llama3.1:8b         → faster, lower quality
# ─────────────────────────────────────────────────────────
TRANSLATION_MODEL = 'translategemma:12b'
REVISION_MODEL    = 'translategemma:12b'

# ─────────────────────────────────────────────────────────
# 🌐 LANGUAGES
# Use ISO 639-1 codes (e.g., 'en', 'pt', 'es', 'fr', 'de', 'it', 'ja')
# SOURCE_LANGUAGE: language of the original subtitles
# TARGET_LANGUAGE: language for the translation output
# ─────────────────────────────────────────────────────────
SOURCE_LANGUAGE = 'en'   # e.g., 'en' = English
TARGET_LANGUAGE = 'pt'   # e.g., 'pt' = Brazilian Portuguese 'fr' = French

# ─────────────────────────────────────────────────────────
# 📁 SRT FILE PATH — change here

# INPUT_FILE = r"C:\path\to\your\file.srt"   # Windows
# INPUT_FILE = "/path/to/your/file.srt"    # Linux/Mac

INPUT_FILE = r"movie_test\BBC.Human.Origins.srt"

# ─────────────────────────────────────────────────────────
# 🔄 TRANSLATION MODE
# True  = Hybrid: Google as base + local model to naturalize (recommended)
# False = Pure model: translation and review done entirely by local model
USE_GOOGLE_AS_BASE = True

# ─────────────────────────────────────────────────────────
# 🎨 TRANSLATION STYLE (see table above)
CHOSEN_STYLE = "1"

# ─────────────────────────────────────────────────────────
# ⚙️ Subtitles per batch
# Recommended: 10 (Academic), 20 (Cinema/Formal), 30 (Colloquial/Casual)
BATCH_SIZE = 20

# ─────────────────────────────────────────────────────────
# 📏 Dynamic character limit per line
CHARS_THRESHOLD = 47    # limit only applied if source line is longer than this
EXPANSION_FACTOR = 1.20  # target limit = source chars × 1.20

# ─────────────────────────────────────────────────────────
# Automatic checks
try:
    available_models = [m.model for m in ollama.list().models]
    print(f'✅ Ollama connected! Models: {available_models}')
except Exception as e:
    raise RuntimeError(f'❌ Ollama not found: {e}. Run: ollama serve')

def _download_model(name):
    print(f'\n📥 Downloading {name}...')
    for p in ollama.pull(name, stream=True):
        status = getattr(p, 'status', '') or ''
        total  = getattr(p, 'total', 0) or 0
        done   = getattr(p, 'completed', 0) or 0
        msg    = f'  {status} {int(done/total*100)}%' if total else f'  {status}'
        print(f'\r{msg}', end='', flush=True)
    print(f'\n✅ {name} ready!')

for model in set([TRANSLATION_MODEL, REVISION_MODEL]):
    if model not in available_models:
        _download_model(model)
    else:
        print(f'✅ Model {model} already installed.')

try:
    GOOGLE_AVAILABLE = bool(GoogleTranslator(source=SOURCE_LANGUAGE, target=TARGET_LANGUAGE).translate('hello'))
    print('✅ Google Translate available.')
except Exception as e:
    GOOGLE_AVAILABLE = False
    print(f'⚠️  Google Translate unavailable: {e}')

style    = STYLES[CHOSEN_STYLE]
mode_str = '🔄 Hybrid' if (USE_GOOGLE_AS_BASE and GOOGLE_AVAILABLE) else '🤖 Pure model'
print(f'\n🎨 Style    : {style["name"]}')
print(f'🔄 Mode     : {mode_str}')
print(f'🌐 Languages: {SOURCE_LANGUAGE} → {TARGET_LANGUAGE}')
print(f'📁 File     : {INPUT_FILE}')
print(f'⚙️  Batches  : {BATCH_SIZE} subtitles | Limit: source > {CHARS_THRESHOLD} → target ≤ source × {EXPANSION_FACTOR}')


In [ ]:
# ✅ Cell 3 — Translate
from functions.translation_process import translate_file

output_filename = translate_file(
    input_file         = INPUT_FILE,
    chosen_style       = CHOSEN_STYLE,
    translation_model  = TRANSLATION_MODEL,
    revision_model     = REVISION_MODEL,
    batch_size         = BATCH_SIZE,
    chars_threshold    = CHARS_THRESHOLD,
    expansion_factor   = EXPANSION_FACTOR,
    use_google_as_base = USE_GOOGLE_AS_BASE,
    google_available   = GOOGLE_AVAILABLE,
    source_language    = SOURCE_LANGUAGE,
    target_language    = TARGET_LANGUAGE,
)


In [ ]:
# ✅ Cell 4 — Final review + Automatic correction (optional)
# Compares the original SRT with the translated one, automatically corrects
# any errors found and saves a new file.
# Run after Cell 3.
from functions.translation_process import review_and_correct_file, STYLES

REVIEW_BATCH_SIZE = 20
SAVE_REPORT       = True

# ─────────────────────────────────────────────────────────
# 🎯 TRANSQUEST — Quality Estimation (optional)
# Detects translations that are semantically wrong/unrelated to the source
# using the TransQuest multilingual DA model (score 0–100, higher = better).
# Requires: pip install transquest  (downloads ~440 MB model on first run)
# ─────────────────────────────────────────────────────────
USE_TRANSQUEST = True   # set True to enable
QE_THRESHOLD   = 50.0   # flag pairs with DA score below this value
                         # 40 = lenient | 50 = balanced | 60 = strict

try:
    _ = output_filename
except NameError:
    raise RuntimeError('❌ Run Cell 3 before this cell.')

output_corrected = review_and_correct_file(
    input_file        = INPUT_FILE,
    output_filename   = output_filename,
    revision_model    = REVISION_MODEL,
    translation_model = TRANSLATION_MODEL,
    style             = STYLES[CHOSEN_STYLE],
    hybrid_mode       = USE_GOOGLE_AS_BASE and GOOGLE_AVAILABLE,
    chars_threshold   = CHARS_THRESHOLD,
    expansion_factor  = EXPANSION_FACTOR,
    review_batch_size = REVIEW_BATCH_SIZE,
    save_report       = SAVE_REPORT,
    use_transquest    = USE_TRANSQUEST,
    qe_threshold      = QE_THRESHOLD,
    source_language   = SOURCE_LANGUAGE,
    target_language   = TARGET_LANGUAGE,
)
